# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [ ]:
import pandas as pd
import numpy as np
import os

# Create synthetic evaluation dataset for playbook generation
np.random.seed(42)
n_samples = 100

df = pd.DataFrame({
    'item_id': [f'content_{i:03d}' for i in range(n_samples)],
    'staleness_days': np.random.randint(5, 90, size=n_samples),
    'ctr_drop': np.random.uniform(0.0, 0.4, size=n_samples),
    'conversion_rate': np.random.uniform(0.01, 0.10, size=n_samples)
})

# Define action mapping and reason codes
def assign_action(row):
    if row['staleness_days'] > 45 and row['ctr_drop'] > 0.20:
        return pd.Series(['FULL_REFRESH', 'HIGH_STALENESS_AND_CTR_DROP', 0.90])
    elif row['ctr_drop'] > 0.15:
        return pd.Series(['TITLE_THUMBNAIL_UPDATE', 'CTR_DROP_ONLY', 0.75])
    elif row['staleness_days'] > 60:
        return pd.Series(['CONTENT_AUDIT', 'HIGH_STALENESS_ONLY', 0.60])
    else:
        return pd.Series(['MONITOR', 'STABLE_METRICS', 0.30])

df[['action_label', 'reason_code', 'priority_score']] = df.apply(assign_action, axis=1)
ranked_playbook = df.sort_values(by='priority_score', ascending=False).reset_index(drop=True)

print("--- Top 5 Action Playbook Items ---")
print(ranked_playbook[['item_id', 'action_label', 'reason_code', 'priority_score']].head())

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

In [ ]:
# Markdown / Comment Documentation of System Boundaries

"""
INTENDED USE:
- Decision-support playbook for content team optimization.
- Prioritizes content decay candidates based on observed engagement and staleness signals.

LIMITS & BOUNDARIES:
- Not an automated publishing pipeline (requires human validation).
- Performs poorly on highly seasonal topics where temporary traffic dips occur naturally.
"""

print("Intended use and system limits documented successfully.")

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

In [ ]:
# Define No-Go Automation Rules
def human_review_gate(row):
    # Rule: High-converting content or high priority items must be human-reviewed
    if row['conversion_rate'] > 0.07 or row['action_label'] == 'FULL_REFRESH':
        return 'MANDATORY_HUMAN_REVIEW'
    return 'AUTOMATED_SUGGESTION'

ranked_playbook['review_status'] = ranked_playbook.apply(human_review_gate, axis=1)

print("--- No-Go & Human Review Distribution ---")
print(ranked_playbook['review_status'].value_counts())

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [ ]:
# Define Data Drift & Retrain Logic
print("--- Monitoring & Retrain Protocol ---")
print("1. Data Drift Trigger: Perform PSI (Population Stability Index) check monthly. If PSI > 0.25 on CTR features, trigger retrain.")
print("2. Performance Decay: If precision drops below 0.75 on human-reviewed decisions over 30 days, pause automation.")
print("3. Retrain Frequency: Scheduled bi-monthly retraining on updated client interaction history.")

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [ ]:
# Export output CSV and metrics JSON to work/outputs/
os.makedirs('work/outputs', exist_ok=True)
os.makedirs('work/figures', exist_ok=True)

# Export ranked queue CSV
ranked_playbook.to_csv('work/outputs/action_playbook_queue.csv', index=False)

# Export summary metrics JSON
summary_metrics = {
    "total_items_analyzed": int(len(ranked_playbook)),
    "full_refresh_count": int((ranked_playbook['action_label'] == 'FULL_REFRESH').sum()),
    "mandatory_human_reviews": int((ranked_playbook['review_status'] == 'MANDATORY_HUMAN_REVIEW').sum())
}

import json
with open('work/outputs/playbook_metrics.json', 'w') as f:
    json.dump(summary_metrics, f, indent=4)

print("Outputs successfully exported to work/outputs/ directory.")

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.